<a href="https://colab.research.google.com/github/NAMUORI00/aerospace-rag/blob/main/notebooks/aerospace_rag_colab_ui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aerospace Local RAG Notebook UI

이 노트북은 항공우주 RAG 파이프라인의 재현성 구조를 단계별로 확인하기 위한 실행 문서입니다. Colab에서는 Google Drive를 전혀 mount하거나 재사용하지 않고, 실행할 때마다 GitHub에서 프로젝트를 새로 clone합니다. 각 섹션은 다음 단계가 의존하는 산출물과 설정을 출력하도록 구성되어 있습니다.

## 1. 실행 환경 확인

현재 런타임이 Colab인지 로컬인지, Python/OS/GPU 상태가 무엇인지 먼저 기록합니다. 이 정보는 같은 노트북을 다시 실행했을 때 환경 차이를 비교하는 기준입니다.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def current_working_dir() -> Path:
    try:
        return Path.cwd()
    except FileNotFoundError:
        fallback = Path("/content") if Path("/content").exists() else Path.home()
        os.chdir(fallback)
        return Path.cwd()


RUN_STARTED_AT = time.strftime("%Y-%m-%d %H:%M:%S %Z", time.localtime())
print("Run started:", RUN_STARTED_AT)
print("Runtime:", "Colab" if IN_COLAB else "Local")
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
        stderr=subprocess.STDOUT,
    ).strip()
except Exception:
    gpu_info = "No GPU reported by nvidia-smi"
print("GPU:", gpu_info)
print("Initial cwd:", current_working_dir())

## 2. 프로젝트 소스 확보

Colab에서는 Google Drive를 사용하지 않습니다. 항상 `/content`로 이동한 뒤 기존 `/content/aerospace-rag`를 삭제하고 GitHub repo를 새로 clone합니다. clone 후 commit hash를 출력해 이번 실행이 어떤 코드 상태였는지 남깁니다.

Google Drive는 사용하지 않습니다.

In [ ]:
GITHUB_REPO_URL = "https://github.com/NAMUORI00/aerospace-rag.git"
DEFAULT_COLAB_ROOT = Path("/content/aerospace-rag")


def ensure_valid_cwd() -> Path:
    try:
        return Path.cwd()
    except FileNotFoundError:
        fallback = DEFAULT_COLAB_ROOT.parent if DEFAULT_COLAB_ROOT.parent.exists() else Path.home()
        os.chdir(fallback)
        return Path.cwd()


def project_root_candidates() -> list[Path]:
    cwd = ensure_valid_cwd()
    return [cwd, cwd.parent]


def is_project_root(path: Path) -> bool:
    return (path / "aerospace_rag").is_dir() and (path / "notebooks").is_dir()


def find_project_root() -> Path | None:
    for candidate in project_root_candidates():
        if is_project_root(candidate):
            return candidate
    return None


def git_output(*args: str, default: str = "unknown") -> str:
    try:
        return subprocess.check_output(["git", *args], text=True, stderr=subprocess.STDOUT).strip()
    except Exception:
        return default


PROJECT_ROOT = None

if IN_COLAB:
    os.chdir(DEFAULT_COLAB_ROOT.parent)
    print("Policy: Google Drive is not mounted or reused.")
    print("Running git clone:", GITHUB_REPO_URL)
    if DEFAULT_COLAB_ROOT.exists():
        shutil.rmtree(DEFAULT_COLAB_ROOT)
    subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(DEFAULT_COLAB_ROOT)])
    PROJECT_ROOT = DEFAULT_COLAB_ROOT if is_project_root(DEFAULT_COLAB_ROOT) else None
else:
    PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. Colab git clone 또는 로컬 프로젝트 경로를 확인하세요."
    )

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
REPO_BRANCH = git_output("rev-parse", "--abbrev-ref", "HEAD")
REPO_COMMIT = git_output("rev-parse", "HEAD")
print("Project root:", PROJECT_ROOT)
print("In Colab:", IN_COLAB)
print("Repo URL:", GITHUB_REPO_URL)
print("Git branch:", REPO_BRANCH)
print("Git commit:", REPO_COMMIT)

## 3. 의존성 설치와 버전 고정 확인

필요한 Python 패키지를 설치하고, 재현성 비교에 필요한 핵심 패키지 버전을 출력합니다. Colab/Linux에서는 Docker 없이 graph channel을 쓰기 위해 `falkordblite`를 선택적으로 설치합니다.

In [ ]:
def package_version(package: str) -> str:
    try:
        return importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"


def ensure_dependencies() -> None:
    required = {
        "qdrant_client": "qdrant-client",
        "falkordb": "falkordb",
        "openpyxl": "openpyxl",
        "pypdf": "pypdf",
        "ipywidgets": "ipywidgets",
        "nbformat": "nbformat",
    }
    missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
    if missing:
        print("Installing:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Base dependencies already installed")

    if importlib.util.find_spec("redislite") is None and IN_COLAB:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "falkordblite>=0.9"])
            print("FalkorDBLite installed")
        except Exception as exc:
            print("FalkorDBLite optional install skipped:", exc)


ensure_dependencies()

PACKAGE_SNAPSHOT = {
    package: package_version(package)
    for package in [
        "qdrant-client",
        "falkordb",
        "falkordblite",
        "openpyxl",
        "pypdf",
        "ipywidgets",
        "nbformat",
    ]
}
print(json.dumps(PACKAGE_SNAPSHOT, ensure_ascii=False, indent=2))

## 4. Ollama 런타임과 모델 준비

LLM 호출은 Ollama `gemma4:e2b`를 기본값으로 사용합니다. Colab CPU에서는 설치와 모델 pull, 첫 응답이 오래 걸릴 수 있으므로 이 단계를 독립적으로 확인합니다.

In [ ]:
os.environ.setdefault("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
os.environ.setdefault("OLLAMA_MODEL", "gemma4:e2b")
os.environ.setdefault("OLLAMA_API_KEY", "")


def ollama_headers() -> dict:
    headers = {"Content-Type": "application/json"}
    if os.environ.get("OLLAMA_API_KEY"):
        headers["Authorization"] = "Bearer " + os.environ["OLLAMA_API_KEY"]
    return headers


def is_ollama_cloud_runtime() -> bool:
    return os.environ["OLLAMA_BASE_URL"].rstrip("/") == "https://ollama.com"


def ollama_api_ok() -> bool:
    try:
        req = urllib.request.Request(
            os.environ["OLLAMA_BASE_URL"].rstrip("/") + "/api/tags",
            headers=ollama_headers(),
        )
        with urllib.request.urlopen(req, timeout=5) as response:
            return response.status == 200
    except Exception:
        return False


def mark_ollama_unavailable(reason: str) -> None:
    print("Ollama unavailable; generation backend remains Ollama.")
    print("ask() will try Ollama first and return extractive evidence if the call fails.")
    print("Reason:", reason)


def install_colab_ollama_prerequisites() -> None:
    if IN_COLAB and shutil.which("zstd") is None:
        print("Installing Ollama prerequisite: zstd")
        subprocess.check_call(["apt-get", "install", "-y", "zstd"])


def ensure_ollama_runtime(pull_model: bool = True) -> None:
    model = os.environ["OLLAMA_MODEL"]
    if is_ollama_cloud_runtime():
        if ollama_api_ok():
            print("Ollama cloud ready:", os.environ["OLLAMA_BASE_URL"], "model:", model)
        else:
            mark_ollama_unavailable("Ollama cloud API did not respond on " + os.environ["OLLAMA_BASE_URL"])
        return

    if not IN_COLAB:
        print("Local runtime: ensure Ollama is running separately.")
        print("Expected:", os.environ["OLLAMA_BASE_URL"], "model:", model)
        return

    if shutil.which("ollama") is None:
        print("Installing Ollama...")
        try:
            install_colab_ollama_prerequisites()
            subprocess.check_call("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
        except Exception as exc:
            mark_ollama_unavailable(f"Ollama install failed: {exc}")
            return

    if not ollama_api_ok():
        print("Starting Ollama server...")
        try:
            subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
        except Exception as exc:
            mark_ollama_unavailable(f"Ollama server start failed: {exc}")
            return
        for _ in range(60):
            if ollama_api_ok():
                break
            time.sleep(1)

    if not ollama_api_ok():
        mark_ollama_unavailable("Ollama server did not become ready on " + os.environ["OLLAMA_BASE_URL"])
        return

    if pull_model:
        print("Pulling Ollama model:", model)
        try:
            subprocess.check_call(["ollama", "pull", model])
        except Exception as exc:
            mark_ollama_unavailable(f"Ollama model pull failed: {exc}")
            return

    print("Ollama ready:", os.environ["OLLAMA_BASE_URL"], "model:", model)


ensure_ollama_runtime(pull_model=True)

## 5. 데이터 파일 준비

`data/` 폴더를 재귀적으로 훑어 지원 형식의 문서를 자동으로 찾습니다. 특정 파일명을 강제하지 않습니다. Colab에서는 파일 패널의 `/content/aerospace-rag/data` 폴더에 업무 문서를 넣은 뒤 이 셀부터 다시 실행하면 됩니다. 파일이 준비되면 파일 크기와 SHA-256 해시를 출력해 같은 데이터로 재현했는지 확인합니다.

In [ ]:
from aerospace_rag.ingestion import SUPPORTED_SUFFIXES, iter_supported_files

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def data_relative_name(path: Path) -> str:
    return str(path.relative_to(DATA_DIR)).replace("\\", "/")


DATA_FILES = list(iter_supported_files(DATA_DIR))
print("데이터 경로:", DATA_DIR)
print("지원 형식:", ", ".join(sorted(SUPPORTED_SUFFIXES)))
if not DATA_FILES:
    print("위 경로에 지원 문서 파일을 넣은 뒤 이 셀을 다시 실행하세요.")
    print("Colab 파일 패널에서 aerospace-rag/data 폴더로 파일을 드래그해 넣으면 됩니다.")
    raise FileNotFoundError(f"지원되는 data 파일이 없습니다: {DATA_DIR}")

DATA_MANIFEST = []
for path in DATA_FILES:
    entry = {
        "name": data_relative_name(path),
        "bytes": path.stat().st_size,
        "sha256": file_sha256(path),
    }
    DATA_MANIFEST.append(entry)
    print(f"OK {entry['name']} bytes={entry['bytes']} sha256={entry['sha256'][:16]}...")
print("Data files:", len(DATA_MANIFEST))

## 6. 실행 설정 확정

이번 실행에서 사용할 RAG 설정을 한곳에 모읍니다. Colab smoke run이 안정적으로 끝나도록 임베딩은 hash fallback을 기본으로 두며, 실제 BGE-M3를 쓰려면 `requirements-models.txt` 설치 후 `AEROSPACE_EMBED_BACKEND=sentence_transformers`로 바꾸면 됩니다.

In [ ]:
os.environ.setdefault("DAT_MODE", "hybrid")
os.environ.setdefault("AEROSPACE_EMBED_BACKEND", "hash")
INDEX_DIR = DATA_DIR / "index"

RUNTIME_CONFIG = {
    "DAT_MODE": os.environ.get("DAT_MODE"),
    "LLM_PROVIDER": "ollama",
    "OLLAMA_BASE_URL": os.environ.get("OLLAMA_BASE_URL"),
    "OLLAMA_MODEL": os.environ.get("OLLAMA_MODEL"),
    "OLLAMA_API_KEY_SET": bool(os.environ.get("OLLAMA_API_KEY")),
    "AEROSPACE_EMBED_BACKEND": os.environ.get("AEROSPACE_EMBED_BACKEND"),
    "INDEX_DIR": str(INDEX_DIR),
}
print(json.dumps(RUNTIME_CONFIG, ensure_ascii=False, indent=2))

## 7. 수집/파싱 단독 확인

인덱스를 만들기 전에 입력 파일이 어떤 chunk로 변환되는지 확인합니다. 이 단계가 통과하면 데이터 파일명, 파일 형식, 기본 파서가 정상이라는 뜻입니다.

In [ ]:
from collections import Counter
from aerospace_rag.ingestion import ingest_data

chunks = ingest_data(DATA_DIR, strict_expected=False)
CHUNK_SUMMARY = {
    "total_chunks": len(chunks),
    "by_file": dict(Counter(chunk.source_file for chunk in chunks)),
    "by_modality": dict(Counter(chunk.modality for chunk in chunks)),
}
print(json.dumps(CHUNK_SUMMARY, ensure_ascii=False, indent=2))

for chunk in chunks[:5]:
    loc = f"p.{chunk.page}" if chunk.page else f"{chunk.sheet}:{chunk.row}" if chunk.row else "table"
    preview = chunk.text.replace("\n", " ")[:240]
    print(f"- {chunk.chunk_id} [{chunk.source_file} / {loc} / {chunk.modality}]")
    print(" ", preview)

## 8. 인덱스 생성

동일 데이터에서 Qdrant local mode, BM25, FalkorDB/FalkorDBLite helper index를 다시 생성합니다. `reset=True`라 이전 index 산출물은 재사용하지 않습니다.

In [ ]:
from aerospace_rag.pipeline import build_index

INDEX_RESET = True
build_started = time.perf_counter()
result = build_index(data_dir=DATA_DIR, index_dir=INDEX_DIR, reset=INDEX_RESET, strict_expected=False)
INDEX_BUILD_SECONDS = round(time.perf_counter() - build_started, 3)

BUILD_REPORT = {
    "data_dir": str(result.data_dir),
    "index_dir": str(result.index_dir),
    "file_count": result.file_count,
    "chunk_count": result.chunk_count,
    "qdrant_collection": result.qdrant_collection,
    "falkordb_path": str(result.falkordb_path),
    "bm25_path": str(result.bm25_path),
    "chunks_path": str(result.chunks_path),
    "reset": INDEX_RESET,
    "seconds": INDEX_BUILD_SECONDS,
}
print(json.dumps(BUILD_REPORT, ensure_ascii=False, indent=2))

## 9. 검색 단독 검증

LLM을 호출하기 전에 retrieval만 실행합니다. 답변 품질 문제가 생겼을 때 검색 문제인지 생성 문제인지 분리해서 볼 수 있습니다.

In [ ]:
from aerospace_rag.config import Settings
from aerospace_rag.stores.local_index import LocalIndex

RETRIEVAL_QUESTION = "위성영상 가격은 저장영상과 신규촬영에서 어떻게 다른가?"
index = LocalIndex(INDEX_DIR, settings=Settings.from_env())
hits = index.search(RETRIEVAL_QUESTION, top_k=5)
print("Question:", RETRIEVAL_QUESTION)
print("Hits:", len(hits))
for i, hit in enumerate(hits, start=1):
    loc = f"p.{hit.chunk.page}" if hit.chunk.page else f"{hit.chunk.sheet}:{hit.chunk.row}" if hit.chunk.row else "table"
    print(f"[{i}] score={hit.score:.3f} channels={hit.channels} {hit.chunk.source_file} ({loc})")
    print(hit.chunk.text.replace("\n", " ")[:420])
    print()
print("Diagnostics:")
print(json.dumps(index.last_diagnostics, ensure_ascii=False, indent=2))

## 10. LLM 답변 생성

동일 질문을 `ask()` 파이프라인으로 실행합니다. Ollama 호출이 실패하면 코드가 extractive 답변으로 fallback하므로, diagnostics에서 provider와 retrieval 상태를 함께 확인합니다.

In [ ]:
from aerospace_rag.pipeline import ask

question = RETRIEVAL_QUESTION
response = ask(question, index_dir=INDEX_DIR, top_k=5, provider=None, debug=True)
print(response.answer)
print("\nRouting:", response.routing)
print("\nDiagnostics:")
print(json.dumps(response.diagnostics, ensure_ascii=False, indent=2))

## 11. 근거 확인

최종 답변에 사용된 source reference를 확인합니다. page/sheet/row와 channel score가 함께 출력됩니다.

In [ ]:
for i, source in enumerate(response.sources, start=1):
    loc = f"p.{source.page}" if source.page else f"{source.sheet}:{source.row}" if source.row else "table"
    print(f"[{i}] {source.source_file} ({loc}) score={source.score:.3f} channels={source.channels}")
    print(source.excerpt[:500])
    print()

## 12. 반복 질문 예시

여러 질문을 같은 index에 대해 실행해 retrieval/answer 흐름이 반복 가능하게 동작하는지 확인합니다.

In [ ]:
sample_questions = [
    "H3 8호기 발사 실패 원인은?",
    "NASA solar sail 연구의 목적은?",
    "국가별 우주항공 예산과 인력 현황은?",
    "인공위성 판매대행사 선정 절차는 얼마나 걸리는가?",
]

SAMPLE_RESULTS = []
for q in sample_questions:
    r = ask(q, index_dir=INDEX_DIR, top_k=4, provider=None, debug=True)
    SAMPLE_RESULTS.append({
        "question": q,
        "source_count": len(r.sources),
        "sources": [s.source_file for s in r.sources[:3]],
        "provider": r.diagnostics.get("provider"),
    })
    print("QUESTION:", q)
    print(r.answer.splitlines()[0] if r.answer else "")
    print("sources:", [s.source_file for s in r.sources[:3]])
    print("-" * 80)

## 13. 실제 업무파일 RAG 검증

실제 `data/` 업무파일 전체로 만든 index에 대해 대표 질문을 실행합니다. 각 질문마다 provider, 검색 채널, top source, 고유 source 목록, 답변 preview를 출력해 Colab에서도 ingest부터 답변 생성까지 한 번에 검증할 수 있습니다.

In [ ]:
ACTUAL_RAG_QUESTIONS = [
    "H3 8호기 발사 경과의 핵심 내용은 무엇인가?",
    "NASA가 Momentus에 수여한 계약은 무엇인가?",
    "위성영상 가격에서 저장영상과 신규촬영은 어떻게 다른가?",
    "해외정부 우주항공 현황 문서에서 확인되는 주요 국가나 기관은 무엇인가?",
    "인공위성 질문응답 문서 기준으로 인공위성 관련 핵심 설명은 무엇인가?",
]

ACTUAL_RAG_RESULTS = []
for case_idx, q in enumerate(ACTUAL_RAG_QUESTIONS, start=1):
    r = ask(q, index_dir=INDEX_DIR, top_k=5, provider=None, debug=True)
    source_files = []
    for source in r.sources:
        if source.source_file not in source_files:
            source_files.append(source.source_file)
    record = {
        "case": case_idx,
        "question": q,
        "provider": r.routing.get("provider"),
        "source_count": len(r.sources),
        "channels": r.diagnostics.get("channels"),
        "fusion": r.diagnostics.get("fusion"),
        "source_files": source_files,
        "top_source": r.sources[0].source_file if r.sources else None,
        "top_score": round(r.sources[0].score, 4) if r.sources else None,
        "answer_preview": r.answer[:900],
    }
    ACTUAL_RAG_RESULTS.append(record)
    print(f"CASE {case_idx}: {q}")
    print("provider:", record["provider"], "source_count:", record["source_count"])
    print("channels:", record["channels"])
    print("top_source:", record["top_source"], "top_score:", record["top_score"])
    print("source_files:", record["source_files"])
    print("answer_preview:", record["answer_preview"].replace("\n", " ")[:500])
    print("-" * 80)

print(json.dumps(ACTUAL_RAG_RESULTS, ensure_ascii=False, indent=2))

## 14. 재현성 리포트

이번 실행을 다시 추적할 수 있도록 코드 commit, 런타임 설정, 패키지 버전, 데이터 파일 manifest, index 산출물 위치, 실제 업무파일 RAG 검증 결과를 한 번에 출력합니다.

In [ ]:
REPRODUCIBILITY_REPORT = {
    "run_started_at": RUN_STARTED_AT,
    "runtime": "Colab" if IN_COLAB else "Local",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "repo_url": GITHUB_REPO_URL,
    "repo_branch": REPO_BRANCH,
    "repo_commit": REPO_COMMIT,
    "project_root": str(PROJECT_ROOT),
    "data_manifest": DATA_MANIFEST,
    "chunk_summary": CHUNK_SUMMARY,
    "runtime_config": RUNTIME_CONFIG,
    "package_snapshot": PACKAGE_SNAPSHOT,
    "build_report": BUILD_REPORT,
    "sample_results": SAMPLE_RESULTS,
    "actual_rag_results": ACTUAL_RAG_RESULTS,
}
print(json.dumps(REPRODUCIBILITY_REPORT, ensure_ascii=False, indent=2))

## 15. 문제 해결 체크리스트

- Colab 연결 자체가 안 되면 브라우저 확장, 로그인 계정, 쿠키 차단, 런타임 제한 상태를 먼저 확인합니다.
- 첫 셀을 재실행했는데 cwd 오류가 나면 이 노트북은 `/content`로 이동한 뒤 clone하도록 구성되어 있으므로 런타임을 다시 시작하고 1번부터 실행합니다.
- `git clone`이 실패하면 GitHub repo 공개 여부와 네트워크 상태를 확인합니다.
- 데이터 파일 오류가 나면 `data/` 폴더에 지원 형식의 문서가 들어 있는지 확인합니다.
- Ollama Cloud를 쓰려면 `OLLAMA_BASE_URL=https://ollama.com`, `OLLAMA_MODEL=gemma4:31b`, `OLLAMA_API_KEY`를 먼저 설정한 뒤 4번 셀부터 실행합니다.
- CPU Colab에서 로컬 Ollama 단계가 오래 걸리면 정상일 수 있습니다. retrieval 검증만 보고 싶다면 9번 검색 단독 검증까지만 실행합니다.
- 답변이 이상하면 9번 검색 단독 검증의 top-k 결과를 먼저 보고, 검색 결과가 맞으면 10번 LLM 답변 쪽을 확인합니다.